#1 Plot Signal (ggF and VBF) with Diboson
selection : n_b_jet = 0 
            n_jet >= 2

In [2]:
import ROOT

f = ROOT.TFile("diboson.root")

f.ls()

TFile**		diboson.root	
 TFile*		diboson.root	
  KEY: TTree	AnalysisMiniTree;1	AnalysisMiniTree


In [3]:
tree = f.Get("AnalysisMiniTree")

tree.Print()

******************************************************************************
*Tree    :AnalysisMiniTree: AnalysisMiniTree                                       *
*Entries :   621842 : Total =       330150040 bytes  File  Size =  150222450 *
*        :          : Tree compression factor =   2.20                       *
******************************************************************************
*Br    0 :dsid      : dsid/I                                                 *
*Entries :   621842 : Total  Size=    2500441 bytes  File Size  =      26754 *
*Baskets :      124 : Basket Size=      32000 bytes  Compression=  93.36     *
*............................................................................*
*Br    1 :hhml_subChannelFlavor : hhml_subChannelFlavor/I                    *
*Entries :   621842 : Total  Size=    2502617 bytes  File Size  =     268200 *
*Baskets :      124 : Basket Size=      32000 bytes  Compression=   9.32     *
*.............................................

In [4]:
f_sig_1 =ROOT.TFile("signal_ggF.root")

f_sig_1.ls()

TFile**		signal_ggF.root	
 TFile*		signal_ggF.root	
  KEY: TTree	AnalysisMiniTree;1	AnalysisMiniTree


In [5]:
tree_sig_1 = f_sig_1.Get("AnalysisMiniTree")

tree.Print()

******************************************************************************
*Tree    :AnalysisMiniTree: AnalysisMiniTree                                       *
*Entries :   621842 : Total =       330150040 bytes  File  Size =  150222450 *
*        :          : Tree compression factor =   2.20                       *
******************************************************************************
*Br    0 :dsid      : dsid/I                                                 *
*Entries :   621842 : Total  Size=    2500441 bytes  File Size  =      26754 *
*Baskets :      124 : Basket Size=      32000 bytes  Compression=  93.36     *
*............................................................................*
*Br    1 :hhml_subChannelFlavor : hhml_subChannelFlavor/I                    *
*Entries :   621842 : Total  Size=    2502617 bytes  File Size  =     268200 *
*Baskets :      124 : Basket Size=      32000 bytes  Compression=   9.32     *
*.............................................

In [6]:
f_sig_2 = ROOT.TFile("signal_VBF.root")

f_sig_2.ls()

TFile**		signal_VBF.root	
 TFile*		signal_VBF.root	
  KEY: TTree	AnalysisMiniTree;1	AnalysisMiniTree


In [7]:
tree_sig_2 = f_sig_2.Get("AnalysisMiniTree")
tree_sig_2.Print()

******************************************************************************
*Tree    :AnalysisMiniTree: AnalysisMiniTree                                       *
*Entries :    47257 : Total =        25293543 bytes  File  Size =   11338054 *
*        :          : Tree compression factor =   2.23                       *
******************************************************************************
*Br    0 :dsid      : dsid/I                                                 *
*Entries :    47257 : Total  Size=     191492 bytes  File Size  =       3295 *
*Baskets :       21 : Basket Size=      32000 bytes  Compression=  57.90     *
*............................................................................*
*Br    1 :hhml_subChannelFlavor : hhml_subChannelFlavor/I                    *
*Entries :    47257 : Total  Size=     191917 bytes  File Size  =      21920 *
*Baskets :       21 : Basket Size=      32000 bytes  Compression=   8.72     *
*.............................................

In [32]:
#combine the two signal trees
sig_chain = ROOT.TChain("AnalysisMiniTree")
sig_chain.Add("signal_ggF.root")
sig_chain.Add("signal_VBF.root")

#opens and grabs background
f_bkg = ROOT.TFile("diboson.root")
t_bkg = f_bkg.Get("AnalysisMiniTree")

#call RDataFrame as its better for cuts
df_sig = ROOT.RDataFrame(sig_chain)
df_bkg = ROOT.RDataFrame(t_bkg)

#apply cuts
df_sig = df_sig.Filter("n_b_jet == 0 && n_jet >= 2")
df_bkg = df_bkg.Filter("n_b_jet == 0 && n_jet >= 2")

#convert met_met from MeV to GeV
df_sig = df_sig.Define("met_met_GeV", "met_met / 1000.")
df_bkg = df_bkg.Define("met_met_GeV", "met_met / 1000.")

#build histo
h_sig = df_sig.Histo1D(("h_sig", "", 50, 0, 200), "met_met_GeV", "weight")
h_bkg = df_bkg.Histo1D(("h_bkg", "", 50, 0, 200), "met_met_GeV", "weight")

#fill before normalising
h_sig.GetValue()
h_bkg.GetValue()

#normalises to 1
h_sig.Scale(1.0 / h_sig.Integral())
h_bkg.Scale(1.0 / h_bkg.Integral())

c = ROOT.TCanvas()
h_bkg.SetLineColor(ROOT.kPink)
h_bkg.Draw("hist")
h_sig.Draw("hist same")
c.SaveAs("met_sig_vs_bkg.png")

Info in <TCanvas::Print>: png file met_sig_vs_bkg.png has been created
